In [1]:
import sys
!{sys.executable} -m pip install -r ./requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.8/455.8 kB 12.7 MB/s eta 0:00:00a 0:00:01
INFO: pip is looking at multiple versions of tensorflow to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 475.2/475.2 MB 221.6 MB/s eta 0:00:0000:0100:01
INFO: pip is looking at multiple versions of <Python from Requires-Python> to determine which version is compatible with other requirements. This could take a while.
ERROR: Cannot install -r ./requirements.txt (line 10), -r ./requirements.txt (line 8) and protobuf<5 and >=4.21.1 because these package versions have conflicting dependencies.

The conflict is caused by:
    The user requested protobuf<5 and >=4.21.1
    tensorflow 2.15.0 depends on protobuf!=4.21.0, !=4.21.1, !=4.21.2, !=4.21.3, !=4.21.4, !=4.21.5, <5.0.0dev and >=3.20.3
    tf2onnx 1.16.1 depends on protobuf~=3.20

To fix this you could try to:
1. loosen the range of package versions you've specifie

In [ ]:
import boto3
import matplotlib.pyplot as plt
import pandas as pd
from pandas_datareader import data as pdr
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from tensorflow import keras
from tf2onnx import convert
import onnx
import yfinance as yfin
import time
from keras.models import Sequential
from keras.layers import Dense, LSTM, Dropout
from minio import Minio
import os
from kfp import compiler
from kfp import dsl
from kfp.dsl import InputPath, OutputPath

from kfp import kubernetes

In [ ]:
ticker = os.environ.get('TICKER')

In [ ]:
df = yfin.download(tickers=['AAPL'], period='6mo')
#df = yfin.download(tickers=ticker, period='6mo')
dataset = df['Close'].fillna(method='ffill')
dataset = dataset.values.reshape(-1, 1)

In [ ]:
dataset.shape

In [ ]:
scaler = MinMaxScaler(feature_range=(0, 1))
scaler = scaler.fit(dataset)
dataset = scaler.transform(dataset)

In [ ]:
# generate the input and output sequences
n_lookback = 60  # length of input sequences (lookback period)
n_forecast = 30  # length of output sequences (forecast period)

X = []
Y = []

for i in range(n_lookback, len(dataset) - n_forecast + 1):
    X.append(dataset[i - n_lookback: i])
    Y.append(dataset[i: i + n_forecast])

In [ ]:
X = np.array(X)
Y = np.array(Y)


In [ ]:
# fit the model
model = Sequential(name="forecast")
model.add(LSTM(units=50, return_sequences=True, input_shape=(n_lookback, 1)))
model.add(LSTM(units=50))
model.add(Dense(n_forecast))

In [ ]:

model.compile(loss='mean_squared_error', optimizer='adam')
model.fit(X, Y, epochs=100, batch_size=32, verbose=0)

In [ ]:
# generate the forecasts
X_ = dataset[- n_lookback:]  # last available input sequence
X_ = X_.reshape(1, n_lookback, 1)

Y_ = model.predict(X_).reshape(-1, 1)
Y_ = scaler.inverse_transform(Y_)

In [ ]:
# organize the results in a data frame
#df_past = df[['Close']].reset_index()
#df_past.rename(columns={'index': 'Date', 'Close': 'Actual'}, inplace=True)
#df_past['Date'] = pd.to_datetime(df_past['Date'])
#df_past['Forecast'] = np.nan
#df_past['Forecast'].iloc[-1] = df_past['Actual'].iloc[-1]

In [ ]:

#df_future = pd.DataFrame(columns=['Date', 'Actual', 'Forecast'])
#df_future['Date'] = pd.date_range(start=df_past['Date'].iloc[-1] + pd.Timedelta(days=1), periods=n_forecast)
#df_future['Forecast'] = Y_.flatten()
#df_future['Actual'] = np.nan

In [ ]:
#results = pd.concat([df_past, df_future])
#results = results.set_index('Date')

# plot the results
#results.plot(title='IBM')


In [ ]:
import os
model.save("./forecast.keras")

In [ ]:
onnx_model, _ = tf2onnx.convert.from_keras(model)
onnx.save(onnx_model, "./forecast.onnx")

In [ ]:
client = Minio(
    "minio.stock-predict.svc.cluster.local:9000",
    access_key="minioadmin",
    secret_key="minioadmin",
    secure=False
)


In [ ]:
buckets = client.list_buckets()
for bucket in buckets:
    print(bucket.name, bucket.creation_date)

In [ ]:
bucket_name = "models"
source_file = "./forecast.onnx"
destination_file = "forecast.onnx"
client.fput_object(bucket_name, destination_file, source_file)